# ML Assignment 2 — Online Shoppers Purchase Prediction (training notebook)

Trains the 5 classifiers and produces everything the Streamlit app needs:
`model/*.joblib`, `model/metrics.csv`, `model/metrics.md`, `test_data.csv` and a version-pinned `requirements.txt`.

**Steps:** upload the project zip → run all cells → download `training_outputs.zip` at the end.

## 1. Upload the project zip
Zip the `online-shoppers-classifier` folder on your laptop and upload it here.

In [ ]:
import os, zipfile, glob

try:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("/content")
    PROJECT = os.path.dirname(glob.glob("/content/**/train_models.py", recursive=True)[0])
except ImportError:
    # not on Colab (e.g. BITS Virtual Lab Jupyter) — just run from the project folder
    PROJECT = os.getcwd()

os.chdir(PROJECT)
print("working in:", PROJECT)
print(sorted(os.listdir()))

## 2. Environment
Note the Python version — pick the same one on Streamlit Cloud (Advanced settings) when deploying.

In [ ]:
import sys, sklearn, numpy, pandas, joblib, matplotlib, seaborn
print("Python      ", sys.version.split()[0])
for m in (sklearn, numpy, pandas, joblib, matplotlib, seaborn):
    print(f"{m.__name__:12s}", m.__version__)

## 3. Quick look at the data

In [ ]:
import pandas as pd
df = pd.read_csv("data/online_shoppers_intention.csv")
print("shape:", df.shape, "| duplicates:", df.duplicated().sum(), "| missing values:", df.isnull().sum().sum())
print("\nRevenue split:\n", df["Revenue"].value_counts(normalize=True).round(3))
df.head()

In [ ]:
df.describe().T.round(2)

## 4. Train all 5 models
Takes a couple of minutes — SVM with probability outputs is the slowest.

In [ ]:
!python train_models.py

## 5. Results

In [ ]:
import matplotlib.pyplot as plt
results = pd.read_csv("model/metrics.csv").set_index("ML Model Name")
display(results)
results.plot.bar(figsize=(11, 4.5), rot=0, title="Test-set metrics by model").legend(loc="lower right", ncol=6)
plt.ylim(0, 1); plt.tight_layout(); plt.show()

Paste this into the README comparison table:

In [ ]:
print(open("model/metrics.md").read())

## 6. Sanity check — reload the saved models exactly like the app does
No retraining here: load each `.joblib` and score `test_data.csv`. Numbers should match the table above.

In [ ]:
import joblib
from shopper_utils import MODEL_FILES, TARGET, clean_features, compute_metrics, to_binary

test = pd.read_csv("test_data.csv")
X_test, y_test = clean_features(test), to_binary(test[TARGET])
check = {}
for name, fname in MODEL_FILES.items():
    model = joblib.load(f"model/{fname}")
    check[name] = compute_metrics(y_test, model.predict(X_test), model.predict_proba(X_test)[:, 1])
pd.DataFrame(check).T.round(4)

## 7. Pin requirements.txt to the versions used for training
This is what keeps the saved models loadable on Streamlit Cloud.

In [ ]:
reqs = [f"scikit-learn=={sklearn.__version__}", f"numpy=={numpy.__version__}", f"pandas=={pandas.__version__}",
        f"joblib=={joblib.__version__}", f"matplotlib=={matplotlib.__version__}", f"seaborn=={seaborn.__version__}",
        "streamlit"]
open("requirements.txt", "w").write("\n".join(reqs) + "\n")
print(open("requirements.txt").read())
print("Use Python", sys.version.split()[0][:4], "on Streamlit Cloud")

## 8. Download outputs
Unzip into your project folder, replacing `requirements.txt`.

In [ ]:
!zip -qr training_outputs.zip model test_data.csv requirements.txt
try:
    files.download("training_outputs.zip")
except NameError:
    print("saved training_outputs.zip in", PROJECT)